In [ ]:
import numpy as np
import matplotlib . pyplot as plt
import os
from matplotlib.colors import LogNorm
import seaborn as sns
import pandas as pd
from scipy.interpolate import griddata

np. random . seed (2025)
n, p, k = 500 , 50, 8
sigma = 1.0

m = 10
Z = np. random . randn (n, m)
A = np. random . randn (m, p) * 0.8
X = Z.dot(A) + 0.1 * np. random . randn (n, p)

w_true = np. zeros (p)
w_true [:k] = np. linspace (5.0 , 1.0 , k) * np. random . choice ([1 , -1], k)
y = X.dot( w_true ) + sigma * np. random . randn (n)

idx = np. random . permutation (n)
train_idx , test_idx = idx [:350] , idx [350:]
X_train , y_train = X[ train_idx ], y[ train_idx ]
X_test , y_test = X[ test_idx ], y[ test_idx ]

def ridge_closed_form (X, y, lam):
    n_features = X.shape[1]
    I = np.eye(n_features)
    w = np.linalg.inv(X.T @ X + lam * I) @ X.T @ y
    return w

def ridge_objective (X, y, w, lam):
    residual = X @ w - y
    return np.dot(residual, residual) + lam * np.dot(w, w)
    

def ridge_grad (X, y, w, lam):
    return 2 * (X.T @ (X @ w - y) + lam * w)

def gradient_descent (X, y, lam , s, alpha , beta ):
    w = np.zeros(X.shape[1])
    obj_values = []

    for _ in range(1000):
        grad = ridge_grad(X, y, w, lam)
        t = s
        while ridge_objective(X, y, w - t*grad, lam) > ridge_objective(X, y, w, lam) - alpha * t * np.dot(grad, grad):
            t *= beta  
        w = w - t * grad
        obj_values.append(ridge_objective(X, y, w, lam))
    return w, obj_values


In [ ]:
lam = 1.0   
s = 1.0     

alpha_values = [0.9, 0.5, 0.1, 0.01]
beta_values = [0.9, 0.5, 0.1, 0.01]
colors = ["darkslateblue", "violet", "darkblue", "purple", "pink", "mediumpurple", "mediumaquamarine", "pink", 'hotpink', 'plum', 
          'lightseagreen', 'lightskyblue', 'mediumorchid', 'cornflowerblue', 'mediumvioletred', 'turquoise']


results = {}

for alpha in alpha_values:
    for beta in beta_values:
        w_gd, obj_values = gradient_descent(X_train, y_train, lam, s, alpha, beta)
        results[(alpha, beta)] = obj_values
        print(f"α={alpha}, β={beta}, final objective={obj_values[-1]:.4f}")

plt.figure(figsize=(10,6))

for i, ((alpha, beta), obj_values) in enumerate(results.items()):
    label = f"α={alpha}, β={beta}"
    iterations = np.arange(1, len(obj_values) + 1)
    plt.plot(iterations, obj_values, label=label, color=colors[i])



plt.xscale('log')   
plt.yscale('log')   
plt.xlabel('Iteration (log)')
plt.ylabel('Objective value (log)')
plt.title('Objective vs Iteration for Different α and β ')
plt.legend()
plt.grid(True, which="both", ls="--", linewidth=0.5)
downloads_path = os.path.join(os.path.expanduser("~"), "Downloads", "plot1.png")



plt.savefig(downloads_path, dpi=1200)
plt.show()


alpha_vals = df_results['alpha'].values
beta_vals = df_results['beta'].values
obj_vals = df_results['final_objective'].values

alpha_lin = np.linspace(alpha_vals.min(), alpha_vals.max(), 50)
beta_lin = np.linspace(beta_vals.min(), beta_vals.max(), 50)
alpha_grid, beta_grid = np.meshgrid(alpha_lin, beta_lin)

obj_grid = griddata(
    points=(alpha_vals, beta_vals),
    values=obj_vals,
    xi=(alpha_grid, beta_grid),
    method='cubic'
)

plt.figure(figsize=(8, 6))
sns.set_theme()  

plt.imshow(
    obj_grid.T, origin='lower',
    extent=[alpha_lin.min(), alpha_lin.max(), beta_lin.min(), beta_lin.max()],
    aspect='auto',
    cmap='cool'
)

plt.colorbar(label='Final Objective Value')
plt.xlabel('alpha')
plt.ylabel('beta')
plt.title('Final Objective Value Heatmap (Interpolated)')
plt.grid(False)  
downloads_path = os.path.join(os.path.expanduser("~"), "Downloads", "final_value_heatmap.png")
plt.savefig(downloads_path, dpi=1200)
plt.show()

def log_tick_formatter(val, pos=None):
    return r"$10^{%.0f}$" % val

alpha_beta_list = [
    (0.01, 0.5),
    (0.01, 0.9),
    (0.9, 0.5),
    (0.9, 0.9),
]

lambdas = [0.01, 0.1, 1, 10, 100, 1000]
s = 1.0
max_iter = 300

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()  


colors = ["mediumvioletred", "slateblue", "lightskyblue", "purple", "lightseagreen", "plum"]   

for i, (alpha, beta) in enumerate(alpha_beta_list):
    ax = axes[i]
    
    for j, lam in enumerate(lambdas):
        norms = gradient_descent_full_trace(X_train, y_train, lam, s, alpha, beta, max_iter=max_iter)
        iters = np.arange(1, len(norms) + 1)

        if colors is None:
            ax.plot(np.log10(iters), norms, label=f"λ={lam}")
        else:
            ax.plot(np.log10(iters), norms, color=colors[j], label=f"λ={lam}")

    ax.set_title(f"(α={alpha}, β={beta})")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("‖wₖ‖₂")
    ax.grid(True)

    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(log_tick_formatter)

    ax.legend()

plt.tight_layout()
downloads_path = os.path.join(os.path.expanduser("~"), "Downloads", "vary_lambda.png")
plt.savefig(downloads_path, dpi=1200)
plt.show()

